# Compare Feature Extraction vs Fine-Tuning step by step

# Step 1 - Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Dense , Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

2024-10-26 18:25:13.889974: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Step 2 -Load and Preprocess the data

In [2]:
# Load CIFAR10 dataset
(x_train , y_train) , (x_test , y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
y_train = to_categorical(y_train , 10)
y_test = to_categorical(y_test , 10)

# Step 3-Load Pretrained Models

**Feature Extraction Using Pretrained VGG16**
In feature extraction, we freeze all the layers of the pretrained model
and only train the final classification layer. Load Pretrained Model
(VGG16): We'll load VGG16 pretrained on ImageNet and freeze its
layers.

In [4]:
# Feature Extraction with VGG16
base_model = VGG16(weights = 'imagenet' , include_top = False , input_shape = (32 , 32 , 3))
for layer in base_model.layers:
    layer.trainable = False

2024-10-26 18:34:48.537314: I tensorflow/core/common_runtime/process_util.cc:146] Creating new thread pool with default inter op setting: 2. Tune using inter_op_parallelism_threads for best performance.


58889256/58889256 [==============================] - 44s 1us/step


**Add New Classification Layers:** We'll add a new fully connected (Dense)
layer for classification on the CIFAR-10 dataset.

In [5]:
x = Flatten()(base_model.output)
x = Dense(512 , activation = 'relu')(x)
output = Dense(10 , activation = 'softmax')(x)

model_feature_extraction = Model(inputs = base_model.input , outputs = output)
model_feature_extraction.compile(optimizer = 'adam' , loss = 'categorical_crossentropy' , metrics = ['accuracy'])

**Train the Model:** Train only the new layers added for classification, keeping the pretrained layers frozen.

In [8]:
history_feature_extraction = model_feature_extraction.fit(x_train , y_train , epochs = 10 , validation_data = (x_test , y_test) , batch_size = 32)

Epoch 1/10
1563/1563 [==============================] - 1278s 818ms/step - loss: 1.2307 - accuracy: 0.5677 - val_loss: 1.2169 - val_accuracy: 0.5723
Epoch 2/10
1563/1563 [==============================] - 1195s 765ms/step - loss: 1.1224 - accuracy: 0.6071 - val_loss: 1.1539 - val_accuracy: 0.5902
Epoch 3/10
1563/1563 [==============================] - 1584s 1s/step - loss: 1.0579 - accuracy: 0.6281 - val_loss: 1.1399 - val_accuracy: 0.5946
Epoch 4/10
1563/1563 [==============================] - 1261s 807ms/step - loss: 1.0038 - accuracy: 0.6480 - val_loss: 1.1131 - val_accuracy: 0.6099
Epoch 5/10
 599/1563 [==========>...................] - ETA: 10:36 - loss: 0.9461 - accuracy: 0.6653

KeyboardInterrupt: 

# Fine-Tuning using Pretrained ResNet50

In **fine-tuning**, we unfreeze some of the layers of the pretrained model and retrain them along with the new layers for the new task.

**Load Pretrained Model (ResNet50):** We'll load **ResNet50** pretrained on **ImageNet** and initially freeze all its layers.

In [ ]:
# Fine-Tuning with ResNet50(without top  fully connected layer)
model = ResNet50(weights = 'imagenet' , include_top = False , input_shape = (32 , 32 , 3))
for layer in base_model.layers:
    layer.trainable = False

Add new classification layer for CIFAR10 task

In [ ]:
x = Flatten()(base_model.output)
x = Dense(512 , activation = 'relu')(x)
output = Dense(10 , activation = 'softmax')(x)

model_finetune = Model(inputs = base_model.input , outputs = output)
model_finetune.compile(optimizer = 'adam' , loss = 'categorical_crossentropy' , metrics = ['accuracy'])

**Fine-Tuning last few Layers:** Unfreeze the last 5 layers of ResNet50 model for fine tuning

In [ ]:
# Unfreeze the last 5 layers of ResNet50
for layer in base_model.layers[-5 : ]:
    layer.trainable = True
    
model_finetune.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
                      

**Train the Model:**
Train the model , allowing the last 5 layers and the new
classification layers to update during training.

In [ ]:
history_finetune = model_finetune.fit(x_train , y_train , epochs =10 , validation_data = (x_test , y_test), batch_size = 32)

# Step 4-Evaluate both Models
We’ll now evaluate both the feature extraction model and the fine-tuned model on the test set and compare their performance.

In [ ]:
loss_feature_extraction , acc_feature_extraction =model_feature_extraction.evaluate(x_test , y_test)
loss_finetune , acc_finetune = model_finetune.evaluate(x_test , y_test)

# Step 5-Compare Results
Finally, compare the results of both models:

In [ ]:
print(f"Feature Extraction Accuracy: {acc_feature_extraction * 100:.2f}%")
print(f"Fine-Tuning Accuracy: {acc_finetune * 100:.2f}%")

# Assignment

How would you improve the **fine-tuning** results. Some Hints are
- Changing learning rate
- Prevent overfitting
- Early stopping
- Preprocessing formatting
- Number of Epochs
- Unfreeze more layers
- Different layers
  
Do **feature extraction** with **Resnet50 **compare the results with
custom model and **VGG16**.

Do **finetuning** with **VGG 16** and feature extraction with
**Resnet50**